In [3]:
import pandas as pd
import numpy as np
import datetime


RESOLUTION = 10 # HZ recordings per second
N_DAYS = 1 # days
PAUSE_TIME = 2 # sec


RIDE_DURATION_PER_FLOOR = 10 # sec
TOTAL_SECONDS = N_DAYS * 24 * 60 * 60
ACCELERATION = 1 # 1 / sec / sec
ACCELERATION_TIME = 5
GAMBLING_DURATION = 5 # sec
PEAK_TIME_PROPABILITY = 0.8
LOW_TIME_PROPABILITY = 0.01


acceleration_frames = ACCELERATION_TIME * RESOLUTION # accelerate for 5 seconds
frequency = int(1 / RESOLUTION * 1000)
timestamps = pd.date_range(start="2026-04-30", periods=TOTAL_SECONDS * RESOLUTION, freq=f'{frequency}ms')

# Stockwerke und Fahrzeiten (Beschleunigung + Fahrt + Bremsen)
floors = [0, 1, 2, 3]

def generate_ride(current_floor=0):
    """Generates a new ride based on the current floor.

    Args:
        current_floor (int): The starting floor of the ride

    Returns:
        Timeseries: of acceleration values (np.array)
        Stop floor: from where to start the next ride
        Starting floor: from which the ride starts (as series)
        Stoppint floor: to which the elevator rides (as series)
        Direction: Ride direction (as series)
    """

    # Decide where to ride to
    available_floors = floors.copy()
    _ = available_floors.pop(available_floors.index(current_floor))
    stop_floor = np.random.choice(available_floors)
    floor_distance = stop_floor - current_floor
    ride_duration = abs(floor_distance * RIDE_DURATION_PER_FLOOR)
    direction = 1 if floor_distance > 0 else -1
    steps = int(ride_duration * RESOLUTION)

    # Beschleunigungsprofil (1s Beschleunigen, Fahrt, 1s Bremsen)
    accel = np.zeros(steps) + np.random.normal(0, 0.25, steps)
    accel[0:acceleration_frames] = ACCELERATION * direction   # Start
    accel[-acceleration_frames:] = -ACCELERATION * direction  # Stopp

    # Richtung (Auf oder Ab)
    start = [current_floor] * steps
    dest = [stop_floor] * steps
    direction = [direction] * steps

    return accel, int(stop_floor), start, dest, direction


# The elevator starts on the ground level in the morning
starting_floor = 0

# Daten-Container
z_acceleration = np.zeros(len(timestamps))
x_acceleration = np.random.normal(20, 0.5, len(timestamps))
y_acceleration = np.random.normal(20, 0.5, len(timestamps))
from_floor = np.zeros(len(timestamps))
to_floor = np.zeros(len(timestamps))
movement_direction = np.zeros(len(timestamps))

# Zeit-Loop für Fahrten
current_step = 0
while current_step < len(timestamps) - 200:
    current_time = timestamps[current_step]
    hour = current_time.hour
    minute = current_time.minute

    # Bestimmung der Wahrscheinlichkeit basierend auf Stoßzeiten
    is_peak = hour in [8, 12, 13] or (hour == 16 and minute >= 30) or (hour == 17 and minute <= 0)
    prob = PEAK_TIME_PROPABILITY if is_peak else LOW_TIME_PROPABILITY

    if np.random.random() < prob:
        ride, starting_floor, ff, tf, md = generate_ride(starting_floor)
        end_step = current_step + len(ride)
        if end_step < len(z_acceleration):
            z_acceleration[current_step:end_step] = ride
            from_floor[current_step:end_step] = ff
            to_floor[current_step:end_step] = tf
            movement_direction[current_step:end_step] = md
        current_step += len(ride) + int(PAUSE_TIME * RESOLUTION) #
    else:
        current_step += int(GAMBLING_DURATION * RESOLUTION)

# Rauschen hinzufügen
z_acceleration += np.random.normal(0, 0.05, len(z_acceleration))

# DataFrame erstellen und speichern
df = pd.DataFrame({
    'timestamp': timestamps,
    'acceleration_x': x_acceleration,
    'acceleration_y': y_acceleration,
    'acceleration_z': z_acceleration,
    'from_floor': from_floor,
    'to_floor': to_floor,
    'movement_direction': movement_direction
})

file_name = f'elevator_{N_DAYS}D_{RESOLUTION}HZ.csv'
df.to_csv(file_name, index=False)
print(f"Datei '{file_name}' wurde erfolgreich erstellt.")


Datei 'elevator_1D_10HZ.csv' wurde erfolgreich erstellt.
